In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# ==========================================
# STEP 1: Load and Clean Telco Churn Dataset
# ==========================================
# Load the dataset from the local environment path
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f"Dataset successfully loaded with {df.shape[0]} rows and {df.shape[1]} columns.")

# Drop unique identifier column as it has no predictive value
df = df.drop('customerID', axis=1)

# Clean 'TotalCharges' column (some rows contain whitespace strings instead of numbers)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Encode target variable: 'Yes' -> 1 (Churned), 'No' -> 0 (Retained)
df['Churn'] = df['Churn'].apply(lambda x: 1 if str(x).strip() == 'Yes' else 0)

# ==========================================
# STEP 2: Feature Engineering & Preprocessing
# ==========================================
# Identify categorical columns and apply One-Hot Encoding
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

# Split data into training (80%) and testing (20%) sets with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale numerical features for model stability
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# STEP 3: Model Training (Random Forest)
# ==========================================
model = RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42)
model.fit(X_train_scaled, y_train)

# ==========================================
# STEP 4: Evaluation & Real-Time Inference
# ==========================================
y_pred = model.predict(X_test_scaled)

print(f"\nModel Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

# Real-time inference function for a new customer profile
def predict_customer_churn(customer_data_dict):
    input_df = pd.DataFrame([customer_data_dict])
    input_encoded = pd.get_dummies(input_df)

    # Reindex to match all training features, filling missing columns with 0
    input_encoded = input_encoded.reindex(columns=X.columns, fill_value=0)

    # Scale and predict
    input_scaled = scaler.transform(input_encoded)
    prediction = model.predict(input_scaled)[0]
    probability = model.predict_proba(input_scaled)[0][1] * 100

    status = "CHURN" if prediction == 1 else "RETAINED"
    return status, probability

# Test Inference with a sample customer profile
sample_customer = {
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'Yes',
    'Dependents': 'No',
    'tenure': 2,
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'Yes',
    'StreamingMovies': 'Yes',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 89.85,
    'TotalCharges': 180.50
}

status, prob = predict_customer_churn(sample_customer)
print(f"\nTest Customer Prediction: {status} ({prob:.1f}% churn probability)")

Dataset successfully loaded with 7043 rows and 21 columns.

Model Accuracy: 80.48%

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.67      0.53      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.80      0.80      0.80      1409


Test Customer Prediction: CHURN (75.9% churn probability)
